# IMDb Top Rated Indian Movies Web Scraping

## Project Objective

The objective of this project is to scrape IMDb's Top Rated Indian Movies using Selenium and BeautifulSoup. The collected data will be used for exploratory data analysis and visualization.

### Data Collected

- Movie Name
- Release Date
- Genre
- Runtime
- Story
- IMDb Rating
- Vote Count
- Director
- Actors
- Movie URL

### Tools Used

- Python
- Selenium
- BeautifulSoup
- Pandas
- JSON
- Jupyter Notebook

## Step 1: Import Required Libraries

In [1]:
import json
import html
import re
import time
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Step 2: Launch Microsoft Edge Browser

In [3]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from webdriver_manager.microsoft import EdgeChromiumDriverManager

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup

import pandas as pd
import time

In [4]:
service = Service(EdgeChromiumDriverManager().install())

driver = webdriver.Edge(service=service)

driver.maximize_window()

print("Edge Opened Successfully")

Edge Opened Successfully


In [5]:
driver.get("https://www.imdb.com/india/top-rated-indian-movies/")

time.sleep(5)

print(driver.title)

Human Verification


## Step 3: Open IMDb Top Rated Indian Movies Page

In [6]:
url = "https://www.imdb.com/india/top-rated-indian-movies/"

driver.get(url)

## Step 4: Collect Movie Links

In [7]:
WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "a"))
)

soup = BeautifulSoup(driver.page_source, "lxml")

movie_links = []

for a in soup.find_all("a", href=True):

    href = a["href"]

    if href.startswith("/title/tt"):

        link = "https://www.imdb.com" + href.split("?")[0]

        if link not in movie_links:
            movie_links.append(link)

movie_links = movie_links[:100]

print("Total Movie Links Collected:", len(movie_links))

Total Movie Links Collected: 100


## Step 5: Preview Movie Links

In [8]:
movie_links[:5]

['https://www.imdb.com/title/tt23849204/',
 'https://www.imdb.com/title/tt0079221/',
 'https://www.imdb.com/title/tt0093603/',
 'https://www.imdb.com/title/tt0367495/',
 'https://www.imdb.com/title/tt10534500/']

## Step 6: Runtime Formatting Function

In [9]:
def format_runtime(runtime):

    if not runtime:
        return ""

    hours = re.search(r'(\d+)H', runtime)
    minutes = re.search(r'(\d+)M', runtime)

    h = hours.group(1) if hours else "0"
    m = minutes.group(1) if minutes else "0"

    return f"{h}h {m}m"

## Step 7: Create Movie Scraping Function

In [10]:
def scrape_movie(movie_url):

    driver.get(movie_url)

    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.TAG_NAME, "script"))
    )

    soup = BeautifulSoup(driver.page_source, "lxml")

    script = None

    for s in soup.find_all("script", type="application/ld+json"):
        if s.string and '"aggregateRating"' in s.string:
            script = s
            break

    if script is None:
        return None

    data = json.loads(script.string)

    directors = ", ".join(
        d["name"] for d in data.get("director", [])
    )

    actors = ", ".join(
        a["name"] for a in data.get("actor", [])
    )

    movie = {
        "Movie Name": data.get("name", ""),
        "Release Date": data.get("datePublished", ""),
        "Genre": ", ".join(data.get("genre", [])),
        "Runtime": format_runtime(data.get("duration", "")),
        "Story": html.unescape(data.get("description", "")),
        "IMDb Rating": data.get("aggregateRating", {}).get("ratingValue", ""),
        "Vote Count": data.get("aggregateRating", {}).get("ratingCount", ""),
        "Director": directors,
        "Actors": actors,
        "Movie URL": movie_url
    }

    return movie

## Step 8: Test the Scraping Function

In [11]:
movie = scrape_movie(movie_links[1])

movie

{'Movie Name': 'Gol Maal',
 'Release Date': '1979-04-20',
 'Genre': 'Comedy, Romance',
 'Runtime': '2h 0m',
 'Story': "A man's simple lie to secure his job escalates into more complex lies when his orthodox boss gets suspicious.",
 'IMDb Rating': 8.5,
 'Vote Count': 22212,
 'Director': 'Hrishikesh Mukherjee',
 'Actors': 'Amol Palekar, Utpal Dutt, Bindiya Goswami',
 'Movie URL': 'https://www.imdb.com/title/tt0079221/'}

## Step 9: Scrape Top 100 Movies

In [12]:
movies_data = []
failed_movies = []

for i, link in enumerate(movie_links[:100], start=1):

    print(f"Scraping {i}/100")

    try:
        movie = scrape_movie(link)

        if movie:
            movies_data.append(movie)
        else:
            failed_movies.append(link)

    except Exception as e:
        print(f"Error on movie {i}: {e}")
        failed_movies.append(link)

print("\nScraping Completed!")
print("Movies Collected:", len(movies_data))
print("Failed Movies:", len(failed_movies))

Scraping 1/100
Scraping 2/100
Scraping 3/100
Scraping 4/100
Scraping 5/100
Scraping 6/100
Scraping 7/100
Scraping 8/100
Scraping 9/100
Scraping 10/100
Scraping 11/100
Scraping 12/100
Scraping 13/100
Scraping 14/100
Scraping 15/100
Scraping 16/100
Scraping 17/100
Scraping 18/100
Scraping 19/100
Scraping 20/100
Scraping 21/100
Scraping 22/100
Scraping 23/100
Scraping 24/100
Scraping 25/100
Scraping 26/100
Scraping 27/100
Scraping 28/100
Scraping 29/100
Scraping 30/100
Scraping 31/100
Scraping 32/100
Scraping 33/100
Scraping 34/100
Scraping 35/100
Scraping 36/100
Scraping 37/100
Scraping 38/100
Scraping 39/100
Scraping 40/100
Scraping 41/100
Scraping 42/100
Scraping 43/100
Scraping 44/100
Scraping 45/100
Scraping 46/100
Scraping 47/100
Scraping 48/100
Scraping 49/100
Scraping 50/100
Scraping 51/100
Scraping 52/100
Scraping 53/100
Scraping 54/100
Scraping 55/100
Scraping 56/100
Scraping 57/100
Scraping 58/100
Scraping 59/100
Scraping 60/100
Scraping 61/100
Scraping 62/100
Scraping 63/100
S

## Step 10: Create DataFrame and Export CSV

In [13]:
df = pd.DataFrame(movies_data)

df.head()

,Movie Name,Release Date,Genre,Runtime,Story,IMDb Rating,Vote Count,Director,Actors,Movie URL
0,12th Fail,2023-10-27,"Biography, Drama",2h 27m,The real-life story of IPS Officer Manoj Kumar...,8.7,175243,Vidhu Vinod Chopra,"Vikrant Massey, Harish Khanna, Geeta Agrawal S...",https://www.imdb.com/title/tt23849204/
1,Gol Maal,1979-04-20,"Comedy, Romance",2h 0m,A man's simple lie to secure his job escalates...,8.5,22212,Hrishikesh Mukherjee,"Amol Palekar, Utpal Dutt, Bindiya Goswami",https://www.imdb.com/title/tt0079221/
2,Nayakan,1987-10-21,"Crime, Drama",2h 25m,A common man's struggles against a corrupt pol...,8.6,28064,Mani Ratnam,"Kamal Haasan, Saranya Ponvannan, Delhi Ganesh",https://www.imdb.com/title/tt0093603/
3,Anbe Sivam,2003-01-14,"Adventure, Comedy, Drama",2h 40m,"Two men, one young and arrogant, the other dam...",8.6,28465,Sundar C.,"Kamal Haasan, Madhavan, Kiran Rathod",https://www.imdb.com/title/tt0367495/
4,#Home,2021-08-19,"Drama, Family",2h 38m,Oliver Twist (Indrans) wants to be tech-savvy ...,8.7,19965,Rojin Thomas,"Indrans, Sreenath Bhasi, Manju Pillai",https://www.imdb.com/title/tt10534500/


In [14]:
df.to_csv("imdb_top100_indian_movies.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


## Step 11: Close the Browser

In [15]:
driver.quit()

In [16]:
import os

print(os.getcwd())

C:\Users\Priya\Priyaprojectgithub
